# Healthcare Fraud Detection — Results Analysis

Experiment 01 (baseline comparison), 02 (budget ablation), 04 (REINFORCE PoC).  
All results loaded from JSON files in `experiments/*/results/`.

In [ ]:
import json
import os
from pathlib import Path
import statistics

ROOT = Path("..")
EXP01 = ROOT / "experiments" / "01_baseline_comparison" / "results"
EXP02 = ROOT / "experiments" / "02_budget_ablation" / "results"
EXP04 = ROOT / "experiments" / "04_reinforce_poc" / "results"

def load(path):
    with open(path) as f:
        return json.load(f)

print("Paths OK:", EXP01.exists(), EXP02.exists(), EXP04.exists())

## 1. Baseline Comparison — All Agents

In [ ]:
# Load canonical result files
deepseek_cmp  = load(EXP01 / "20260403_230615_comparison.json")
qwen_naive    = load(EXP01 / "20260403_161250_NaiveLLMqwen3.6-plus_free.json")
reinforce_ev  = load(EXP04 / "20260405_000606_eval_ReinforceAgent.json")
random_ev     = load(EXP04 / "20260405_000606_eval_RandomAgent.json")
threshold_ev  = load(EXP04 / "20260405_000606_eval_ThresholdAgent.json")

agents = [
    {
        "name": "RandomAgent",
        "reward":   random_ev["mean_reward"],
        "f1":       random_ev["mean_f1"],
        "recall":   random_ev["mean_recall"],
        "budget":   random_ev["mean_budget_utilization"],
        "mem":      random_ev["mean_memory_reuse_rate"],
        "latency":  random_ev["mean_latency_ms"],
    },
    {
        "name": "ThresholdAgent",
        "reward":   threshold_ev["mean_reward"],
        "f1":       threshold_ev["mean_f1"],
        "recall":   threshold_ev["mean_recall"],
        "budget":   threshold_ev["mean_budget_utilization"],
        "mem":      threshold_ev["mean_memory_reuse_rate"],
        "latency":  threshold_ev["mean_latency_ms"],
    },
    {
        "name": "NaiveLLM (Qwen3.6:free)",
        "reward":   qwen_naive["mean_reward"],
        "f1":       qwen_naive["mean_f1"],
        "recall":   qwen_naive["mean_recall"],
        "budget":   qwen_naive["mean_budget_utilization"],
        "mem":      qwen_naive["mean_memory_reuse_rate"],
        "latency":  None,  # ~8s/call
    },
    {
        "name": "NaiveLLM (DeepSeek V3.2)",
        "reward":   deepseek_cmp["agents"][0]["mean_reward"],
        "f1":       deepseek_cmp["agents"][0]["mean_f1"],
        "recall":   deepseek_cmp["agents"][0]["mean_recall"],
        "budget":   deepseek_cmp["agents"][0]["mean_budget_utilization"],
        "mem":      deepseek_cmp["agents"][0]["mean_memory_reuse_rate"],
        "latency":  deepseek_cmp["agents"][0]["mean_latency_ms"],
    },
    {
        "name": "BudgetAware (DeepSeek V3.2)",
        "reward":   deepseek_cmp["agents"][1]["mean_reward"],
        "f1":       deepseek_cmp["agents"][1]["mean_f1"],
        "recall":   deepseek_cmp["agents"][1]["mean_recall"],
        "budget":   deepseek_cmp["agents"][1]["mean_budget_utilization"],
        "mem":      deepseek_cmp["agents"][1]["mean_memory_reuse_rate"],
        "latency":  deepseek_cmp["agents"][1]["mean_latency_ms"],
    },
    {
        "name": "REINFORCE (linear)",
        "reward":   reinforce_ev["mean_reward"],
        "f1":       reinforce_ev["mean_f1"],
        "recall":   reinforce_ev["mean_recall"],
        "budget":   reinforce_ev["mean_budget_utilization"],
        "mem":      reinforce_ev["mean_memory_reuse_rate"],
        "latency":  reinforce_ev["mean_latency_ms"],
    },
]

# Print table
print(f"{'Agent':<30} {'Reward':>8} {'F1':>6} {'Recall':>7} {'BudgUse':>8} {'MemReuse':>9}")
print("-" * 75)
for a in sorted(agents, key=lambda x: -x["reward"]):
    print(f"{a['name']:<30} {a['reward']:>8.0f} {a['f1']:>6.3f} {a['recall']:>7.1%} "
          f"{a['budget']:>8.1%} {a['mem']:>9.1%}")

## 2. Prompt Improvement: NaiveLLM vs BudgetAware (Same Model)

In [ ]:
naive_r  = deepseek_cmp["agents"][0]["mean_reward"]
budget_r = deepseek_cmp["agents"][1]["mean_reward"]
improvement_pct = (budget_r - naive_r) / abs(naive_r) * 100

print(f"NaiveLLM (DeepSeek):     {naive_r:,.0f}")
print(f"BudgetAware (DeepSeek):  {budget_r:,.0f}")
print(f"Improvement:             {budget_r - naive_r:+,.0f} reward ({improvement_pct:+.1f}%)")
print(f"Multiplier:              {abs(naive_r)/abs(budget_r):.2f}x better reward")

## 3. Budget Ablation — Rule-Based Agents

In [ ]:
ablation = load(EXP02 / "20260404_233454_ablation_summary.json")

print(f"{'Budget':>8} {'Random Reward':>15} {'Threshold Reward':>17} {'Random F1':>10} {'Threshold F1':>13}")
print("-" * 68)
for b in ablation["budgets_tested"]:
    data = ablation["by_budget"][str(b)]
    rand = next(d for d in data if "Random" in d["name"])
    thr  = next(d for d in data if "Threshold" in d["name"])
    print(f"{b:>8} {rand['mean_reward']:>15.0f} {thr['mean_reward']:>17.0f} "
          f"{rand['mean_f1']:>10.3f} {thr['mean_f1']:>13.3f}")

print()
print("Key: ThresholdAgent reward is perfectly flat (never uses INVESTIGATE).")
print("     RandomAgent gets WORSE with more budget — more random investigations.")

## 4. REINFORCE Training Curve

In [ ]:
curve = load(EXP04 / "20260405_000606_training_curve.json")
summary = load(EXP04 / "20260405_000606_summary.json")

# Rolling average — take every 20th entry (window stored in data)
episodes = [e["episode"] for e in curve]
rewards  = [e["total_reward"] for e in curve]
rolling  = [e.get("mean_reward_window", e["total_reward"]) for e in curve]
entropy  = [e.get("policy_entropy", None) for e in curve]

print(f"Training episodes:   {len(curve)}")
print(f"First 25 ep avg:     {statistics.mean(rewards[:25]):,.1f}")
print(f"Last 25 ep avg:      {statistics.mean(rewards[-25:]):,.1f}")
print(f"Q1 mean reward:      {summary['learning']['first_quartile_mean_reward']:,.1f}")
print(f"Q4 mean reward:      {summary['learning']['last_quartile_mean_reward']:,.1f}")
print(f"Total improvement:   +{summary['learning']['improvement']:,.1f}")
print(f"Training time:       {summary['training_time_s']:.1f}s")

# Plot if matplotlib available
try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1.plot(episodes, rewards, alpha=0.2, color='steelblue', linewidth=0.5, label='Episode reward')
    ax1.plot(episodes, rolling, color='steelblue', linewidth=2, label='Rolling avg (20)')
    ax1.axhline(-799, color='green', linestyle='--', linewidth=1.5, label='ThresholdAgent benchmark')
    ax1.axhline(-2173, color='red', linestyle='--', linewidth=1.5, label='RandomAgent baseline')
    ax1.set_ylabel('Episode Reward')
    ax1.set_title('REINFORCE Training — Healthcare Fraud Detection')
    ax1.legend(loc='lower right')
    ax1.grid(alpha=0.3)

    valid_entropy = [(e, v) for e, v in zip(episodes, entropy) if v is not None]
    if valid_entropy:
        ep_e, ent_e = zip(*valid_entropy)
        ax2.plot(ep_e, ent_e, color='orange', linewidth=1.5)
        ax2.set_ylabel('Policy Entropy')
        ax2.set_xlabel('Episode')
        ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../blog/reinforce_training_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: blog/reinforce_training_curve.png")
except ImportError:
    print("matplotlib not installed — skipping plot")

## 5. Key Numbers for Blog Post

In [ ]:
print("=" * 60)
print("KEY NUMBERS FOR BLOG POST")
print("=" * 60)

best_llm   = budget_r
worst_llm  = qwen_naive["mean_reward"]
random_r   = random_ev["mean_reward"]
thresh_r   = threshold_ev["mean_reward"]

print(f"\n1. LLM vs Random:")
print(f"   NaiveLLM Qwen ({worst_llm:,.0f}) is WORSE than Random ({random_r:,.0f})")
print(f"   by {worst_llm - random_r:,.0f} reward points")

print(f"\n2. Prompt engineering uplift (same DeepSeek model):")
print(f"   Naive: {naive_r:,.0f}  ->  BudgetAware: {best_llm:,.0f}")
print(f"   = {abs(naive_r)/abs(best_llm):.2f}x better, +{best_llm-naive_r:,.0f} reward")

print(f"\n3. Rule-based vs best LLM:")
print(f"   ThresholdAgent ({thresh_r:,.0f}) vs BudgetAware ({best_llm:,.0f})")
print(f"   Threshold is still {thresh_r - best_llm:,.0f} worse (BudgetAware wins!)")

print(f"\n4. REINFORCE improvement:")
print(f"   +{summary['learning']['improvement']:,.0f} reward over 500 episodes ({summary['training_time_s']:.0f}s)")

print(f"\n5. Budget paradox (RandomAgent):")
b_data = ablation["by_budget"]
r5  = next(d for d in b_data["5"]  if "Random" in d["name"])["mean_reward"]
r20 = next(d for d in b_data["20"] if "Random" in d["name"])["mean_reward"]
print(f"   Budget=5:  {r5:,.0f}  |  Budget=20: {r20:,.0f}")
print(f"   More budget = {r20-r5:,.0f} worse reward for random agent")

## 6. Reward Distribution per Agent (Box Plot)

In [ ]:
try:
    import matplotlib.pyplot as plt

    def episode_rewards(data):
        return [ep["total_reward"] for ep in data["episodes"]]

    box_data = {
        "Random": episode_rewards(random_ev),
        "Threshold": episode_rewards(threshold_ev),
        "NaiveLLM\n(Qwen)": [ep["total_reward"] for ep in qwen_naive["episodes"]],
        "NaiveLLM\n(DeepSeek)": [],  # need per-episode from individual file
        "BudgetAware\n(DeepSeek)": [],
        "REINFORCE": episode_rewards(reinforce_ev),
    }
    # Load individual DeepSeek files for per-episode data
    try:
        ds_naive  = load(EXP01 / "20260403_230615_NaiveLLMdeepseek-v3.2.json")
        ds_budget = load(EXP01 / "20260403_230615_BudgetAwaredeepseek-v3.2.json")
        box_data["NaiveLLM\n(DeepSeek)"]  = episode_rewards(ds_naive)
        box_data["BudgetAware\n(DeepSeek)"] = episode_rewards(ds_budget)
    except FileNotFoundError:
        pass

    labels = [k for k, v in box_data.items() if v]
    values = [v for v in box_data.values() if v]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(values, labels=labels, patch_artist=True, notch=False)

    colors = ['#d73027', '#4dac26', '#fdae61', '#f46d43', '#74add1', '#313695']
    for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.set_ylabel('Episode Reward (higher = better)')
    ax.set_title('Reward Distribution by Agent — 20 Episodes Each')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('../blog/reward_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: blog/reward_distribution.png")
except ImportError:
    print("matplotlib not installed — skipping plot")